# Breast Cancer Prediction Pipeline

This notebook implements the main pipeline for breast cancer classification using RNA-Seq data. It includes data loading and preprocessing, statistical feature selection (ANOVA F-test), dimensionality reduction (PCA), and classification using a Random Forest model.

## Setup and Configuration

This step sets up the required libraries and defines the folder structure used in this notebook. The data is organized into four main folders under `data/`:
- `raw/`: Raw downloaded files (e.g., GTEx `.gct`, GDC `.tsv`).
- `initial/`: Processed raw files (e.g., merged and transposed).
- `interim/`: Feature-selected or partially transformed files.
- `processed/`: Final model-ready inputs (e.g., PCA-reduced features).

In [1]:
# Import required libraries
import os
import pandas as pd
import numpy as np
import seaborn as sns
import shutil
import pickle
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, permutation_test_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.metrics import roc_curve, auc

In [2]:
# Define data folder structure
DATA_DIR = "data"
RAW_GDC_DIR = os.path.join(DATA_DIR, "raw_gdc_data")
RAW_GTX_DIR = os.path.join(DATA_DIR, "raw_gtx_data")

INITIAL_DIR = os.path.join(DATA_DIR, "initial")
INTERIM_DIR = os.path.join(DATA_DIR, "interim")

MODELS_DIR = "models"
REPORTS_DIR = "reports"

In [ ]:
# Create directories if they don't exist
os.makedirs(INITIAL_DIR, exist_ok=True)
os.makedirs(INTERIM_DIR, exist_ok=True)

os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

### Step 1: Initial Process of GDC Raw Sample Files

This block processes RNA-Seq files downloaded from the GDC portal. Each `.tsv` file contains gene expression data for a single cancer sample. From each file, the column `tpm_unstranded` is extracted, indexed by `gene_id`.

These per-sample files are merged into a single matrix where:
- **Rows** = genes,
- **Columns** = samples,
- **Values** = TPM expression values.

This process results in a file `gdc_data.csv` saved to the `data/initial/` folder. 

It is computationally expensive, so it is designed to be skipped once completed unless the raw data changes.

#### Step 1.1: Merge All Sample Files
Merging separate cancer sample files and Building a complete dataset file

In [ ]:
# Define paths for GDC sample sheet and data files
gdc_sample_sheet_path = os.path.join(RAW_GDC_DIR, 'gdc_sample_sheet.tsv')
gdc_merged_file_path = os.path.join(INITIAL_DIR, 'gdc_data.csv')

In [ ]:
sample_sheet_file_handler = pd.read_csv(gdc_sample_sheet_path, sep='\t')

# Initialize an empty DataFrame to store the combined data
gdc_df = pd.DataFrame()

sample_counter = 1
# Iterate through each file listed in the sample sheet
for index, row in sample_sheet_file_handler.iterrows():
    folder_id = row['File ID']
    file_name = row['File Name']
    
    # Construct the full file path
    sample_file_path = os.path.join(RAW_GDC_DIR, folder_id, file_name)
    
    # Read the TSV file
    try:
        sample_data = pd.read_csv(sample_file_path, sep='\t', skiprows=[0, 2, 3, 4, 5])
        
        # Extract the relevant columns ('gene_id' and 'TPM' or equivalent)
        relevant_data = sample_data[['gene_id', 'tpm_unstranded']]
        
        # Rename the columns to match the GTex format
        sample_number = 'c_' + str(sample_counter).zfill(4)
        relevant_data.columns = ['gene_id', sample_number]  # Use folder_id as the sample name
        
        # Merge with the combined data
        if gdc_df.empty:
            gdc_df = relevant_data
        else:
            gdc_df = pd.merge(gdc_df, relevant_data, on='gene_id', how='outer')

        sample_counter += 1
            
    except Exception as e:
        print(f"Error processing file {sample_file_path}: {e}")

gdc_df.to_csv(gdc_merged_file_path, index=False)

In [ ]:
# Load and display the first few rows of the merged GDC dataset
gdc_merged_file_path = os.path.join(INITIAL_DIR, 'gdc_data.csv')

gdc_merged_file_df = pd.read_csv(gdc_merged_file_path, index_col=0)
print("Shape of initial GDC Data Set", gdc_merged_file_df.shape)
gdc_merged_file_df.head(5)

#### Step 1.2: Split into Model Training Data and Unseen Testing Data
Building a model training dataset of `train_num` samples and an unseen dataset of `unseen_num` samples for testing fo deploying model.

Default Values of GDC tcga data for Breast Cancer: 
- Total number of samples: `total_num = 1231`
- Model Training `train_num= 1000`
- Unseen Testing Data `unseen_num = 231`

In [3]:
# Define path
gdc_data_file_path = os.path.join(INITIAL_DIR, 'gdc_data.csv')

gdc_df = pd.read_csv(gdc_data_file_path, nrows=5)
columns = gdc_df.columns.tolist()

gene_info_columns = columns[0:1]
sample_columns = columns[1:]

In [4]:
# Split data into data for model building and training and unseen data for testing
total_num = len(sample_columns)         # bc tcga data = 1231
train_num = 1000
unseen_num = total_num - train_num

In [6]:
# Randomly select 1000 sample columns from the dataset as traiing data
chosen_columns_for_training = np.random.choice(sample_columns, train_num, replace=False).tolist()
final_columns_for_training = gene_info_columns + chosen_columns_for_training

# Get the remaining columns as testing data
chosen_columns_for_testing = [col for col in sample_columns if col not in chosen_columns_for_training]
final_columns_for_testing = gene_info_columns + chosen_columns_for_testing

In [ ]:
# Load the dataset again but only with the selected columns
gdc_train_df = pd.read_csv(gdc_data_file_path, usecols=final_columns_for_training)
gdc_unseen_df = pd.read_csv(gdc_data_file_path, usecols=final_columns_for_testing)

In [ ]:
# Save the training and testing data to new files
gdc_training_file_path = os.path.join(INITIAL_DIR, 'gdc_data_training.csv')
gdc_testing_file_path = os.path.join(INITIAL_DIR, 'gdc_data_testing.csv')

gdc_train_df.to_csv(gdc_training_file_path, index=False)
gdc_unseen_df.to_csv(gdc_testing_file_path, index=False)

In [ ]:
print("Shape of initial GDC Training Data Set", gdc_train_df.shape)
gdc_train_df.head(5)

In [ ]:
print("Shape of initial GDC Testing Data Set", gdc_unseen_df.shape)
gdc_unseen_df.head(5)

### Step 2: Initial Process of GTEx Raw Sample Files

This step processes transcriptomic data from GTEx. The original file is in `.gct` format and includes expression levels for thousands of genes across thousands of samples.

This step includes:
1. **Conversion of `.gct` to `.csv`**, skipping initial metadata rows.
2. **Random sampling of 1000 columns (samples)** to ensure balance with the GDC dataset.
3. Saving the final dataset as `gtx_data.csv` in `data/initial/`.

Like GDC, this step is time-consuming and should be skipped in repeated runs unless data needs to be refreshed.

In [ ]:
# Define paths
gtx_17K_gct_file_path = os.path.join(RAW_GTX_DIR, 'gtx_raw_data.gct')
gtx_17K_csv_file_path = os.path.join(RAW_GTX_DIR, 'gtx_raw_data.csv')

##### Step 2.1: Convert GCT to CSV
This block defines the `read_gct()` function, which skips the top two header lines from the `.gct` file and loads the remaining expression matrix into a DataFrame. The `convert_gct_to_csv()` function wraps this logic and saves the result as a temporary `.csv` file.

In [ ]:
def read_gct(file_path):
    with open(file_path, 'r') as f:
        # Skip the first two header lines
        for _ in range(2):
            next(f)
        # Read the rest of the file into a pandas DataFrame
        df = pd.read_csv(f, sep='\t')
    return df

In [ ]:
def convert_gct_to_csv(gct_file, csv_file):
    df = read_gct(gct_file)
    df.to_csv(csv_file, index=False)

In [ ]:
convert_gct_to_csv(gtx_17K_gct_file_path, gtx_17K_csv_file_path)

In [ ]:
# Define path
gtx_17K_csv_file_path = os.path.join(RAW_GTX_DIR, 'gtx_raw_data.csv')
gtx_data_file_path = os.path.join(INITIAL_DIR, 'gtx_data.csv')

In [ ]:
gtx_17K_df = pd.read_csv(gtx_17K_csv_file_path, nrows=5)
columns = gtx_17K_df.columns.tolist()

# The first two columns are 'Name' and 'Description', we keep them and sample the rest
gene_info_columns = columns[0:2]
sample_columns = columns[2:]

# Randomly select 1000 sample columns from the dataset as traiing data
chosen_1231_columns = np.random.choice(sample_columns, 1231, replace=False).tolist()
final_1231_columns = gene_info_columns + chosen_1231_columns

# Load the dataset again but only with the selected columns
gtx_1231_df = pd.read_csv(gtx_17K_csv_file_path, usecols=final_1231_columns)
gtx_1231_df = gtx_1231_df.rename(columns={'Name': 'gene_id'})
gtx_1231_df = gtx_1231_df.drop('Description', axis=1)

# Changing the sample ids to h_xxxx format
num_samples = gtx_1231_df.shape[1] - 1
new_sample_names = [f"h_{i:04d}" for i in range(1, num_samples + 1)]
gtx_1231_df.columns = [gtx_1231_df.columns[0]] + new_sample_names


# Save the sampled data to a new CSV file
gtx_1231_df.to_csv(gtx_data_file_path, index=False)

In [ ]:
# Load and display the first few rows of the GTex dataset
gtx_data_df = pd.read_csv(gtx_data_file_path, index_col=0)
print("Shape of initial GTex Data Set", gtx_data_df.shape)
gtx_data_df.head(5)

##### Step 2.2: Randomly sample 1000 columns (samples)
This block randomly selects 1000 GTEx samples (columns) from the large CSV file produced in the previous step. Only gene identifiers and the selected samples are retained. The resulting matrix is saved to `data/initial/gtx_data.csv`.

In [ ]:
gtx_data_df = pd.read_csv(gtx_data_file_path, nrows=5)
columns = gtx_data_df.columns.tolist()

# The first two columns are 'Name' and 'Description', we keep them and sample the rest
gene_info_columns = columns[0:1]
tpm_columns = columns[1:]

# Randomly select 1000 sample columns from the dataset as traiing data
chosen_columns_for_training = np.random.choice(tpm_columns, 1000, replace=False).tolist()
final_columns_for_training = gene_info_columns + chosen_columns_for_training

# Randomly select 231 sample columns from the dataset as traiing data
chosen_columns_for_testing = [col for col in tpm_columns if col not in chosen_columns_for_training]
# remaining_columns = [col for col in tpm_columns if col not in chosen_columns_for_training]
# chosen_columns_for_testing = np.random.choice(remaining_columns, 231, replace=False).tolist()
final_columns_for_testing = gene_info_columns + chosen_columns_for_testing

# Load the dataset again but only with the selected columns
gtx_1000_df = pd.read_csv(gtx_data_file_path, usecols=final_columns_for_training)
# gtx_1000_df = gtx_1000_df.rename(columns={'Name': 'gene_id'})
# gtx_1000_df = gtx_1000_df.drop('Description', axis=1)

gtx_231_df = pd.read_csv(gtx_data_file_path, usecols=final_columns_for_testing)
# gtx_231_df = gtx_231_df.rename(columns={'Name': 'gene_id'})
# gtx_231_df = gtx_231_df.drop('Description', axis=1)

# Save the sampled data to a new CSV file
gtx_training_file_path = os.path.join(INITIAL_DIR, 'gtx_data_training.csv')
gtx_testing_file_path = os.path.join(INITIAL_DIR, 'gtx_data_testing.csv')

gtx_1000_df.to_csv(gtx_training_file_path, index=False)
gtx_231_df.to_csv(gtx_testing_file_path, index=False)


In [ ]:
print("Shape of initial Gtex Training Data Set", gtx_1000_df.shape)
gtx_1000_df.head(5)

In [ ]:
print("Shape of initial GTex Testing Data Set", gtx_231_df.shape)
gtx_231_df.head(5)

### Step 3: Merge and Label Cancer and Non-Cancer Data

This step merges the GTEx (non-cancer) and GDC (cancer) RNA-Seq datasets and prepares them for machine learning. The goal is to align the gene expression profiles from both sources, assign binary labels, and generate a unified dataset.

This step includes:
1. **Loads**:
   - `gtx_data.csv` and `gdc_data.csv` from `data/initial/`
2. **Aligns gene features** (columns) to keep only shared genes between GTEx and GDC.
3. **Transposes the matrices**:
   - Each **row** becomes a sample,
   - Each **column** is a gene (TPM value).
4. **Assigns labels**:
   - `0` for GTEx (healthy samples),
   - `1` for GDC (cancer samples).
5. **Combines** both datasets into a single feature matrix and a corresponding label vector.

##### Output files (in `data/interim/`):
- `preprocessed_data_features.csv`: Combined matrix of all samples with aligned gene features (samples × genes)
- `preprocessed_data_labels.csv`: Binary labels for each sample (0 = GTEx, 1 = GDC)

These files serve as input to the next steps: statistical feature selection and dimensionality reduction.


In [ ]:
def merge_data(gtx_file_path, gdc_file_path):    

    # Load the datasets with headers
    gtx__df = pd.read_csv(gtx_file_path, header=0)
    gdc__df = pd.read_csv(gdc_file_path, header=0)

    # Ensure that the 'gene_id' column is the index for both datasets
    gtx__df.set_index('gene_id', inplace=True)
    gdc__df.set_index('gene_id', inplace=True)

    # Add labels row: 0 for GTEx (healthy) and 1 for GDC (cancer)
    gtx_labels = pd.DataFrame([0] * gtx__df.shape[1], index=gtx__df.columns, columns=['label']).transpose()
    gdc_labels = pd.DataFrame([1] * gdc__df.shape[1], index=gdc__df.columns, columns=['label']).transpose()

    # Concatenate labels and data
    gtx__df = pd.concat([gtx_labels, gtx__df])
    gdc__df = pd.concat([gdc_labels, gdc__df])

    # Combine both datasets
    combined_data = pd.concat([gtx__df, gdc__df], axis=1)

    # Transpose the data to have samples as rows and genes as columns
    combined_data = combined_data.transpose()

    # Separate features and labels
    labels = combined_data['label']
    features = combined_data.drop(columns=['label'])

    # Save preprocessed features and labels to CSV
    output_file_prefix = os.path.join(INTERIM_DIR, 'training_data')
    features.to_csv(output_file_prefix + '_features.csv', index=False)
    labels.to_csv(output_file_prefix + '_labels.csv', index=False)

    print("Data Merging Complete")
    print("Shape of features:", features.shape)
    print("Shape of labels:", labels.shape)

In [ ]:
gtx_data_file = os.path.join(INITIAL_DIR, 'gtx_data_training.csv')
gdc_data_file = os.path.join(INITIAL_DIR, 'gdc_data_training.csv')

merge_data(gtx_data_file, gdc_data_file)